# 0. Problem
## 1661. Average Time of Process per Machine — Easy
For each machine, calculate the average elapsed time between each process `start` and `end`. Round to 3 decimals.
Official: https://leetcode.com/problems/average-time-of-process-per-machine/

# 1. Setup

In [ ]:
import pandas as pd
activity_rows=[(0,0,"start",0.712),(0,0,"end",1.520),(0,1,"start",3.140),(0,1,"end",4.120),(1,0,"start",0.550),(1,0,"end",1.550),(1,1,"start",0.430),(1,1,"end",1.420)]
activity_pd=pd.DataFrame(activity_rows,columns=["machine_id","process_id","activity_type","timestamp"])
activity_pd

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
spark=SparkSession.builder.getOrCreate()
activity_spark=spark.createDataFrame(activity_rows,["machine_id","process_id","activity_type","timestamp"])
activity_spark.createOrReplaceTempView("Activity")

# 2. SQL Solution

In [ ]:
sql_result=spark.sql("""SELECT a.machine_id,ROUND(AVG(b.timestamp-a.timestamp),3) AS processing_time FROM Activity a JOIN Activity b ON a.machine_id=b.machine_id AND a.process_id=b.process_id AND a.activity_type='start' AND b.activity_type='end' GROUP BY a.machine_id ORDER BY a.machine_id""")
sql_result.show(truncate=False)

# 3. pandas Solution

In [ ]:
pivot_pd=activity_pd.pivot(index=["machine_id","process_id"],columns="activity_type",values="timestamp").reset_index()
pivot_pd["elapsed"]=pivot_pd["end"]-pivot_pd["start"]
result_pd=pivot_pd.groupby("machine_id",as_index=False).agg(processing_time=("elapsed","mean")); result_pd["processing_time"]=result_pd["processing_time"].round(3)
result_pd

# 4. PySpark Solution

In [ ]:
starts=activity_spark.filter(F.col("activity_type")=="start").alias("s"); ends=activity_spark.filter(F.col("activity_type")=="end").alias("e")
result_spark=(starts.join(ends,(F.col("s.machine_id")==F.col("e.machine_id"))&(F.col("s.process_id")==F.col("e.process_id"))).groupBy(F.col("s.machine_id").alias("machine_id")).agg(F.round(F.avg(F.col("e.timestamp")-F.col("s.timestamp")),3).alias("processing_time")).orderBy("machine_id")); result_spark.show(truncate=False)

# 5. Pattern Mapping
| Concept | SQL | pandas | PySpark |
|---|---|---|---|
| pair start/end | self join | `.pivot()` | filtered self join |
| elapsed average | `AVG(end-start)` | subtract + `.mean()` | `avg(end-start)` |

# 6. Muscle-Memory Round

พิมพ์ใหม่เองโดยไม่ copy คำตอบด้านบน

In [ ]:
# MUSCLE MEMORY — SQL
# Rebuild using temp view(s): Activity

In [ ]:
# MUSCLE MEMORY — PANDAS
# Rebuild using: activity_pd

In [ ]:
# MUSCLE MEMORY — PYSPARK
# Rebuild using: activity_spark